In [10]:
from langchain_community.document_loaders import TextLoader,JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

import re

In [11]:
input_dirs = Path("./vnexpress_output")
MAX_ARTICLES = 100

documents = []

In [12]:
# Dataset structure: ./data/vnexpress/<slug>/article.json
article_paths = list(input_dirs.glob("**/article.json"))
if not article_paths:
    raise FileNotFoundError(f"No article.json found under: {input_dirs}")

def json_metadata_func(record: dict, metadata: dict ) -> dict:
    record_metadata = {
        "source": record.get("url", ""),
        "title": record.get("title", ""),
        "published_time": record.get("published_time", ""),
        "news": record.get("news", ""),
    }
    return {**metadata, **record_metadata}

for article_path in article_paths[:MAX_ARTICLES]:
    print(f"Processing: {article_path}")
    loader = JSONLoader(
        str(article_path),
        jq_schema= '.',
        content_key="content",
        metadata_func=json_metadata_func
    )
    documents = loader.load()
    if not documents:
        print(f"Warning: No documents loaded from {article_path}")
        continue
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    documents.extend(text_splitter.split_documents(documents))


Processing: vnexpress_output\1-000-chuyen-vien-du-le-ra-quan-du-an-sun-riverpolis-4944171\article.json
Processing: vnexpress_output\1-000-nguoi-trai-nghiem-do-thi-nam-long-qua-thuc-te-ao-4990371\article.json
Processing: vnexpress_output\1-000-tran-cam-quan-cua-pep-guardiola-4964611\article.json
Processing: vnexpress_output\1-1-ty-nen-mua-ford-everest-2023-4955163\article.json
Processing: vnexpress_output\1-200-duoc-si-cap-nhat-ve-ung-dung-probiotics-dang-bao-tu-4995950\article.json
Processing: vnexpress_output\1-500-tien-hien-vat-viet-nam-qua-cac-thoi-ky-4975773\article.json
Processing: vnexpress_output\1-ty-nen-mua-toyota-corolla-cross-hay-vinfast-vf-7-4988611\article.json
Processing: vnexpress_output\2-000-em-nho-chinh-phuc-kun-marathon-hai-phong-4996194\article.json
Processing: vnexpress_output\2-3-trieu-hoc-sinh-ha-noi-nghi-hoc-ngay-1-10-4945669\article.json
Processing: vnexpress_output\2-4-trieu-nguoi-my-nguy-co-mat-tem-phieu-thuc-pham-4988128\article.json
Processing: vnexpress_ou

In [13]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=50)
chunked_documents = text_splitter.split_documents(documents)
print(f"Total documents after splitting: {len(chunked_documents)}") 

print("Sample document:")
print(chunked_documents[0]) if chunked_documents else print("No documents available.")

Total documents after splitting: 10
Sample document:
page_content='TS Hồ Sỹ Hòa, Giám đốc nghiên cứu và tư vấn đầu tư, Công ty CP Chứng khoán DNSE cho rằng, bối cảnh trong nước nhạy cảm với sự phân hóa lớn giữa các doanh nghiệp, nhất là nhóm có vốn hóa lớn. Mặt bằng lãi suất có xu hướng đi lên (lãi suất OMO tăng 50 điểm, lãi suất huy động các ngân hàng thương mại nhích lên dịp cuối năm) buộc nhà đầu tư theo dõi sát các tín hiệu từ quốc tế đến trong nước.

Trên bình diện quốc tế, đầu tháng 11 đến 20/11, chỉ số S&P 500 và Nasdaq liên tiếp giảm do thị trường giảm kỳ vọng trong việc nới lỏng. Tuy nhiên, sau giai đoạn này, hai chỉ số trên bật tăng đáng kể khi dữ liệu kinh tế cải thiện và Fed tiếp tục hạ lãi suất thêm 0,25 điểm phần trăm, qua đó giúp thị trường phản ứng tích cực.' metadata={'source': 'https://vnexpress.net/3-yeu-to-tac-dong-den-thi-truong-chung-khoan-cuoi-nam-4993192.html', 'seq_num': 1, 'title': '3 yếu tố tác động đến thị trường chứng khoán cuối năm', 'published_time': 'Thứ

In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
sample_embedding = embedding.embed_query("hello world")
embedding_dim = len(sample_embedding)
print(f"Embedding dimension: {embedding_dim}")

Embedding dimension: 384


Store in QDrant DB

In [15]:
from dotenv import load_dotenv
import os   

load_dotenv()
qdrant_host = os.getenv("QDRANT_HOST", "localhost")
qdrant_port = int(os.getenv("QDRANT_PORT", 6333))   
collection_name = os.getenv("COLLECTION_NAME", "vnexpress_articles")

In [16]:
from qdrant_client import QdrantClient
from qdrant_client.http import models
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(host=qdrant_host, port=qdrant_port)
if not client.collection_exists(collection_name):
    client.create_collection(
        collection_name=collection_name,
        vectors_config=models.VectorParams(
            size = embedding_dim,
            distance=models.Distance.COSINE
        )
    )   

qdrant = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embedding
)

qdrant.add_documents(chunked_documents)

['87ed19f7a72340589b66de3c1bcafdb1',
 '1337b4d2f4034dd896c19e34f2076bd8',
 '18598c7fbb274e98b28b6b21bf0a9fc1',
 'e3f6b2a538ca48168ed90c41a733223a',
 '29cfef3119514599a4647fecd75c8a09',
 '0b41812e8a3f4c6d8035db26e39f6b35',
 '4d029579cc054aeca1862540fba4d521',
 '206c525d3b28479f891e3e53d23b5217',
 '7ad7f22fd1194ded9e33db3530748336',
 'e2519935add6450881f004c42443b498']